In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import re
from deep_translator import GoogleTranslator
from selenium.webdriver.common.keys import Keys

In [2]:
def to_json(data):
    """
    This function takes list of dictionary as Input and 
    then Creates a JSON file in which Input data is stored
    """
    with open("data_dict.json", "w") as outfile:
        json.dump(data, outfile,indent=4)
        outfile.close()

In [3]:
def get_data(slug_name):
    data_list = []
    url = "https://www.govtrack.us/congress/members/current"
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized") 
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--log-level=3")
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.headless = True
    translator = GoogleTranslator(target='english')
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()
    driver.get(url)
    time.sleep(3)
    try:
        driver.find_element(By.XPATH, f'/html/body/div[3]/div/div/div[3]/button').click()
    except:
        pass
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            break
        last_height = new_height

    list1 = driver.find_elements(By.XPATH, f'/html/body/div[1]/div[4]/div[2]/section/div/div[2]/a')
    print(len(list1))
    for i in range(1, len(list1)+1):
        try:
            data_dict = {}
            referenceUrls = ""
            website = ""
            careerInfoDesignation = ""
            careerInfoStartDate = ""
            careerInfoEndDate =""
            careerInfo = ""
            description = ""
            alias = ""
            image = ""
            twitterUrl = ""
            link = driver.find_element(By.XPATH, f'/html/body/div[1]/div[4]/div[2]/section/div/div[2]/a[{i}]').get_attribute("href")
            driver.execute_script("window.open('');")
            driver.switch_to.window(driver.window_handles[1])
            driver.get(link)
            fullName = driver.find_element(By.XPATH, f'/html/body/div[1]/div[4]/div/div[1]/h1').text
            title = fullName.split(".", 1)[0]
            fullName = fullName.split(".", 1)[1].strip()
            if '“' in fullName:
                firstName = fullName.split('“')[0].strip()
                lastName = fullName.split('” ')[1]
                alias = fullName.split('“')[1].split("”")[0].strip()
                fullName = firstName + " " + lastName
            print(fullName)
            print(alias)
            careerInfoDesignation = driver.find_element(By.XPATH, f'/html/body/div[1]/div[4]/div/div[1]/p[1]').text
            print(careerInfoDesignation)
            try:
                image = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[2]/div/div[1]/div[3]/div[2]/img').get_attribute('src')
                print(image)
            except:
                pass
            try:
                list2 = driver.find_elements(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a')
            except:
                try:
                    list2 = driver.find_elements(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a')
                except:
                    pass
            for j in range(1, len(list2)+1):
                try:
                    heading = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').text
                except:
                    try:
                        heading = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').text
                    except:
                        pass
                if "Website" in heading:
                    website = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').get_attribute('href')
                    print(website)
                elif "@" in heading:
                    twitterUrl = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').get_attribute('href')
                elif "Website" not in heading or "@" not in heading:
                    referenceUrls = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').get_attribute('href') + "; " + referenceUrls
            print(referenceUrls)
            summary = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[2]/div/div[1]/div[1]/div/p[1]').text.replace("(view map)", "")
            try:
                careerInfoStartDate = summary.split("served since")[1].split(".", 1)[0].strip()
                print(careerInfoStartDate)
            except:
                pass
            careerInfoEndDate = summary.split("serves until")[1].replace(".", "").strip()
            print(careerInfoEndDate)
            try:
                description = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[2]/div/div[1]/div[1]/div[2]/div').text
            except:
                pass
            try:
                careerInfo = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[2]/div/div[1]/div[1]/div/p[2]').text
            except:
                pass

            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            if fullName:
                data_dict['fullName'] = fullName
            if alias:
                data_dict['alias'] = alias
            if title:
                data_dict['title'] = title
            if careerInfoDesignation:
                data_dict['careerInfoDesignation'] = careerInfoDesignation
            if description:
                data_dict['description'] = description
            if image:
                data_dict['image'] = image
            if careerInfo:
                data_dict['careerInfo'] = careerInfo
            if careerInfoStartDate:
                data_dict['careerInfoStartDate'] = careerInfoStartDate
            if careerInfoEndDate:
                data_dict['careerInfoEndDate'] = careerInfoEndDate
            if website:
                data_dict['website'] = website
            if twitterUrl:
                data_dict['twitterUrl'] = twitterUrl
            if referenceUrls:
                data_dict['referenceUrls'] = referenceUrls
            if summary:
                data_dict['summary'] = summary
            data_list.append(data_dict)
            print("*"*50)
        except Exception as e:
            print(e)
            pass
    for k in range(1, 1550):
        try:
            data_dict2 = {}
            referenceUrls = ""
            website = ""
            careerInfoDesignation = ""
            careerInfoStartDate = ""
            careerInfoEndDate =""
            careerInfo = ""
            description = ""
            alias = ""
            image = ""
            twitterUrl = ""
            link = driver.find_element(By.XPATH, f'/html/body/div[1]/div[4]/div[2]/section/div/div[2]/div[{k}]/a').get_attribute("href")
            driver.execute_script("window.open('');")
            driver.switch_to.window(driver.window_handles[1])
            driver.get(link)
            fullName = driver.find_element(By.XPATH, f'/html/body/div[1]/div[4]/div/div[1]/h1').text
            title = fullName.split(".", 1)[0]
            fullName = fullName.split(".", 1)[1].strip()
            if '“' in fullName:
                firstName = fullName.split('“')[0].strip()
                lastName = fullName.split('” ')[1]
                alias = fullName.split('“')[1].split("”")[0].strip()
                fullName = firstName + " " + lastName
            print(fullName)
            print(alias)
            careerInfoDesignation = driver.find_element(By.XPATH, f'/html/body/div[1]/div[4]/div/div[1]/p[1]').text
            print(careerInfoDesignation)
            try:
                image = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[2]/div/div[1]/div[3]/div[2]/img').get_attribute('src')
                print(image)
            except:
                pass
            try:
                list2 = driver.find_elements(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a')
            except:
                try:
                    list2 = driver.find_elements(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a')
                except:
                    pass
            for j in range(1, len(list2)+1):
                try:
                    heading = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').text
                except:
                    try:
                        heading = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').text
                    except:
                        pass
                if "Website" in heading:
                    website = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').get_attribute('href')
                    print(website)
                elif "@" in heading:
                    twitterUrl = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').get_attribute('href')
                elif "Website" not in heading or "@" not in heading:
                    referenceUrls = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[1]/div/div[1]/div/a[{j}]').get_attribute('href') + "; " + referenceUrls
            print(referenceUrls)
            summary = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[2]/div/div[1]/div[1]/div/p[1]').text.replace("(view map)", "")
            try:
                careerInfoStartDate = summary.split("served since")[1].split(".", 1)[0].strip()
                print(careerInfoStartDate)
            except:
                pass
            careerInfoEndDate = summary.split("serves until")[1].replace(".", "").strip()
            print(careerInfoEndDate)
            try:
                description = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[2]/div/div[1]/div[1]/div[2]/div').text
            except:
                pass
            try:
                careerInfo = driver.find_element(By.XPATH, f'/html/body/div[1]/div[5]/div/div[2]/div/div[1]/div[1]/div/p[2]').text
            except:
                pass

            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            if fullName:
                data_dict2['fullName'] = fullName
            if alias:
                data_dict2['alias'] = alias
            if title:
                data_dict2['title'] = title
            if careerInfoDesignation:
                data_dict2['careerInfoDesignation'] = careerInfoDesignation
            if description:
                data_dict2['description'] = description
            if image:
                data_dict2['image'] = image
            if careerInfo:
                data_dict2['careerInfo'] = careerInfo
            if careerInfoStartDate:
                data_dict2['careerInfoStartDate'] = careerInfoStartDate
            if careerInfoEndDate:
                data_dict2['careerInfoEndDate'] = careerInfoEndDate
            if website:
                data_dict2['website'] = website
            if twitterUrl:
                data_dict2['twitterUrl'] = twitterUrl
            if referenceUrls:
                data_dict2['referenceUrls'] = referenceUrls
            if summary:
                data_dict2['summary'] = summary
            data_list.append(data_dict2)
        except:
            pass
    driver.quit()
    return data_list
    
        

In [4]:
if __name__ == '__main__':
    data_list = get_data("add_slug_name")
    to_json(data_list)

510
Robert Aderholt

Representative for Alabama’s 4th District
https://www.govtrack.us/static/legislator-photos/400004-200px.jpeg
https://aderholt.house.gov/
http://www.c-spanvideo.org/person/45516; http://bioguide.congress.gov/scripts/biodisplay.pl?index=A000055; http://votesmart.org/candidate/441; http://www.opensecrets.org/politicians/summary.php?cid=N00003028; 
Jan 7, 1997
Jan 3, 2023
**************************************************
Pete Aguilar

House Democratic Caucus Vice Chair and Representative for California’s 31st District
https://www.govtrack.us/static/legislator-photos/412615-200px.jpeg
https://aguilar.house.gov/
http://www.c-spanvideo.org/person/79994; http://bioguide.congress.gov/scripts/biodisplay.pl?index=A000371; http://votesmart.org/candidate/70114; http://www.opensecrets.org/politicians/summary.php?cid=N00033997; 
Jan 6, 2015
Jan 3, 2023
**************************************************
Rick Allen

Representative for Georgia’s 12th District
https://www.govtrack.u

http://www.c-spanvideo.org/person/1031622; http://bioguide.congress.gov/scripts/biodisplay.pl?index=B001267; http://votesmart.org/candidate/110942; http://www.opensecrets.org/politicians/summary.php?cid=N00030608; 
Jan 22, 2009
Jan 3, 2023
**************************************************
Cliff Bentz

Representative for Oregon’s 2nd District
https://www.govtrack.us/static/legislator-photos/456842-200px.jpeg
https://bentz.house.gov/
http://bioguide.congress.gov/scripts/biodisplay.pl?index=B000668; http://www.opensecrets.org/politicians/summary.php?cid=N00045773; 
Jan 3, 2021
Jan 3, 2023
**************************************************
Ami Bera

Representative for California’s 7th District
https://www.govtrack.us/static/legislator-photos/412512-200px.jpeg
https://bera.house.gov/
http://www.c-spanvideo.org/person/1033636; http://bioguide.congress.gov/scripts/biodisplay.pl?index=B001287; http://votesmart.org/candidate/120030; http://www.opensecrets.org/politicians/summary.php?cid=N00030

Jamaal Bowman

Representative for New York’s 16th District
https://www.govtrack.us/static/legislator-photos/456839-200px.jpeg
https://bowman.house.gov/
http://bioguide.congress.gov/scripts/biodisplay.pl?index=B001223; http://www.opensecrets.org/politicians/summary.php?cid=N00044790; 
Jan 3, 2021
Jan 3, 2023
**************************************************
Brendan Boyle

Representative for Pennsylvania’s 2nd District
https://www.govtrack.us/static/legislator-photos/412652-200px.jpeg
https://boyle.house.gov/
http://www.c-spanvideo.org/person/76428; http://bioguide.congress.gov/scripts/biodisplay.pl?index=B001296; http://votesmart.org/candidate/47357; http://www.opensecrets.org/politicians/summary.php?cid=N00035307; 
Jan 3, 2019
Jan 3, 2023
**************************************************
Kevin Brady

Representative for Texas’s 8th District
https://www.govtrack.us/static/legislator-photos/400046-200px.jpeg
https://kevinbrady.house.gov/
http://www.c-spanvideo.org/person/45749; http://b

Maria Cantwell

Senator for Washington
https://www.govtrack.us/static/legislator-photos/300018-200px.jpeg
https://www.cantwell.senate.gov/
http://www.c-spanvideo.org/person/26137; http://bioguide.congress.gov/scripts/biodisplay.pl?index=C000127; http://votesmart.org/candidate/27122; http://www.opensecrets.org/politicians/summary.php?cid=N00007836; 
Jan 3, 2001
Jan 3, 2025
**************************************************
Shelley Capito

Senator for West Virginia
https://www.govtrack.us/static/legislator-photos/400061-200px.jpeg
https://www.capito.senate.gov/
http://www.c-spanvideo.org/person/83737; http://bioguide.congress.gov/scripts/biodisplay.pl?index=C001047; http://votesmart.org/candidate/11701; http://www.opensecrets.org/politicians/summary.php?cid=N00009771; 
Jan 6, 2015
Jan 3, 2027
**************************************************
Salud Carbajal

Representative for California’s 24th District
https://www.govtrack.us/static/legislator-photos/412686-200px.jpeg
https://carbajal.h

http://www.c-spanvideo.org/person/86147; http://bioguide.congress.gov/scripts/biodisplay.pl?index=C001109; http://votesmart.org/candidate/171319; http://www.opensecrets.org/politicians/summary.php?cid=N00035504; 
Jan 3, 2017
Jan 3, 2023
**************************************************
Sheila Cherfilus-McCormick

Representative for Florida’s 20th District
https://cherfilus-mccormick.house.gov/
http://bioguide.congress.gov/scripts/biodisplay.pl?index=C001127; 
Jan 18, 2022
Jan 3, 2023
**************************************************
Judy Chu

Representative for California’s 27th District
https://www.govtrack.us/static/legislator-photos/412379-200px.jpeg
https://chu.house.gov/
http://www.c-spanvideo.org/person/92573; http://bioguide.congress.gov/scripts/biodisplay.pl?index=C001080; http://votesmart.org/candidate/16539; http://www.opensecrets.org/politicians/summary.php?cid=N00030600; 
Jan 3, 2013
Jan 3, 2023
**************************************************
David Cicilline

Represent

...

http://www.c-spanvideo.org/person/71083; http://bioguide.congress.gov/scripts/biodisplay.pl?index=S001195; http://votesmart.org/candidate/59318; http://www.opensecrets.org/politicians/summary.php?cid=N00035282; 
Jun 4, 2013
Jan 3, 2023
**************************************************
Tina Smith

Senator for Minnesota
https://www.govtrack.us/static/legislator-photos/412742-200px.jpeg
https://www.smith.senate.gov/
http://www.c-spanvideo.org/person/111313; http://bioguide.congress.gov/scripts/biodisplay.pl?index=S001203; http://votesmart.org/candidate/152968; http://www.opensecrets.org/politicians/summary.php?cid=N00042353; 
Jan 3, 2018
Jan 3, 2027
**************************************************
Darren Soto

Representative for Florida’s 9th District
https://www.govtrack.us/static/legislator-photos/412695-200px.jpeg
https://soto.house.gov/
http://www.c-spanvideo.org/person/104534; http://bioguide.congress.gov/scripts/biodisplay.pl?index=S001200; http://votesmart.org/candidate/67618; h

http://www.c-spanvideo.org/person/2737; http://bioguide.congress.gov/scripts/biodisplay.pl?index=T000472; http://votesmart.org/candidate/22337; http://www.opensecrets.org/politicians/summary.php?cid=N00006701; 
Jan 3, 2013
Jan 3, 2023
**************************************************
Claudia Tenney

Representative for New York’s 22nd District
https://www.govtrack.us/static/legislator-photos/412720-200px.jpeg
https://tenney.house.gov/
http://www.c-spanvideo.org/person/103481; http://bioguide.congress.gov/scripts/biodisplay.pl?index=T000478; http://votesmart.org/candidate/127668; http://www.opensecrets.org/politicians/summary.php?cid=N00036351; 
Feb 11, 2021
Jan 3, 2023
**************************************************
Jon Tester

Senator for Montana
https://www.govtrack.us/static/legislator-photos/412244-200px.jpeg
https://www.tester.senate.gov/
http://www.c-spanvideo.org/person/1020176; http://bioguide.congress.gov/scripts/biodisplay.pl?index=T000464; http://votesmart.org/candidate/2

Fred Upton

Representative for Michigan’s 6th District
https://www.govtrack.us/static/legislator-photos/400414-200px.jpeg
https://upton.house.gov/
http://www.c-spanvideo.org/person/12127; http://bioguide.congress.gov/scripts/biodisplay.pl?index=U000031; http://votesmart.org/candidate/26906; http://www.opensecrets.org/politicians/summary.php?cid=N00004133; 
Jan 5, 1993
Jan 3, 2023
**************************************************
David Valadao

Representative for California’s 21st District
https://www.govtrack.us/static/legislator-photos/412515-200px.jpeg
https://valadao.house.gov/
http://www.c-spanvideo.org/person/623702; http://bioguide.congress.gov/scripts/biodisplay.pl?index=V000129; http://votesmart.org/candidate/120200; http://www.opensecrets.org/politicians/summary.php?cid=N00033367; 
Jan 3, 2021
Jan 3, 2023
**************************************************
Jefferson Van Drew

Representative for New Jersey’s 2nd District
https://www.govtrack.us/static/legislator-photos/412796-2

http://www.c-spanvideo.org/person/1034044; http://bioguide.congress.gov/scripts/biodisplay.pl?index=W000815; http://votesmart.org/candidate/135326; http://www.opensecrets.org/politicians/summary.php?cid=N00033310; 
Jan 3, 2013
Jan 3, 2023
**************************************************
Bruce Westerman

Representative for Arkansas’s 4th District
https://www.govtrack.us/static/legislator-photos/412610-200px.jpeg
https://westerman.house.gov/
http://www.c-spanvideo.org/person/76097; http://bioguide.congress.gov/scripts/biodisplay.pl?index=W000821; http://votesmart.org/candidate/119120; http://www.opensecrets.org/politicians/summary.php?cid=N00035527; 
Jan 6, 2015
Jan 3, 2023
**************************************************
Jennifer Wexton

Representative for Virginia’s 10th District
https://www.govtrack.us/static/legislator-photos/412834-200px.jpeg
https://wexton.house.gov/
http://bioguide.congress.gov/scripts/biodisplay.pl?index=W000825; http://votesmart.org/candidate/147013; http://

Kathy Castor

Representative for Florida’s 14th District
https://www.govtrack.us/static/legislator-photos/412195-200px.jpeg
https://castor.house.gov/
http://www.c-spanvideo.org/person/1022874; http://bioguide.congress.gov/scripts/biodisplay.pl?index=C001066; http://votesmart.org/candidate/53825; http://www.opensecrets.org/politicians/summary.php?cid=N00027514; 
Jan 3, 2013
Jan 3, 2023
Connie Conway

Representative for California’s 22nd District
http://bioguide.congress.gov/scripts/biodisplay.pl?index=C001128; 
Jun 7, 2022
Jan 3, 2023
Warren Davidson

Representative for Ohio’s 8th District
https://www.govtrack.us/static/legislator-photos/412675-200px.jpeg
https://davidson.house.gov/
http://www.c-spanvideo.org/person/102555; http://bioguide.congress.gov/scripts/biodisplay.pl?index=D000626; http://votesmart.org/candidate/166760; http://www.opensecrets.org/politicians/summary.php?cid=N00038767; 
Jun 9, 2016
Jan 3, 2023
Neal Dunn

Representative for Florida’s 2nd District
https://www.govtra